In [1]:
import json
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon, Point, box
import h3
import ee
import requests
import os
from scipy.spatial import cKDTree
from shapely.geometry import Point
import polars as pl
import time

In [3]:
H3_RESOLUTION = 8
WIND_GEOJSON_PATH = "/content/wind_us_farms.geojson"
SOLAR_GEOJSON_PATH = "/content/solar_us_farms.geojson"

In [4]:
def load_dataset(path: str) -> gpd.GeoDataFrame:
    gdf = gpd.read_file(path)
    # ensure WGS84
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326")
    else:
        gdf = gdf.to_crs("EPSG:4326")
    return gdf


def turbine_points_to_h3(gdf: gpd.GeoDataFrame, resolution: int) -> set[str]:
    """Convert each turbine point to its containing H3 cell index."""
    h3_indices = set()
    for _, row in gdf.iterrows():
        pt = row.geometry
        idx = h3.latlng_to_cell(pt.y, pt.x, resolution)
        h3_indices.add(idx)
    return h3_indices

def generate_conus_h3_cells(resolution: int) -> set[str]:
    """
    Generate all H3 cells covering the contiguous US.

    Strategy: use h3.polygon_to_cells() with a rough CONUS bounding polygon.
    This is the fast way — no need to iterate a manual grid.
    """
    # Rough CONUS boundary (simplified polygon)
    # For production, use a proper US boundary shapefile
    conus_coords = [
        (-125.0, 24.5), (-66.0, 24.5), (-66.0, 49.5),
        (-125.0, 49.5), (-125.0, 24.5)
    ]
    # h3.polygon_to_cells expects an H3 Polygon:
    # outer ring as list of (lat, lng) tuples — NOTE: h3 uses (lat, lng) not (lng, lat)
    outer_ring = [(lat, lng) for lng, lat in conus_coords]

    cells = h3.polygon_to_cells(
        h3.LatLngPoly(outer_ring),
        res=resolution
    )
    return set(cells)

def build_grid_dataframe(
    all_cells: set[str],
    turbine_cells: set[str]
) -> pd.DataFrame:
    """
    Build a DataFrame where each row is an H3 cell with:
      - h3_index: the cell ID
      - lat, lng: cell center
      - has_turbine: 1/0
    """
    records = []
    for cell in all_cells:
        lat, lng = h3.cell_to_latlng(cell)
        records.append({
            "h3_index": cell,
            "lat": lat,
            "lng": lng,
            "has_turbine": int(cell in turbine_cells),
        })
    return pd.DataFrame(records)

In [5]:
import os
print(os.path.getsize(WIND_GEOJSON_PATH))  # compare to expected size

# peek at the end of the file — valid GeoJSON ends with }
with open(WIND_GEOJSON_PATH, 'rb') as f:
    f.seek(-50, 2)
    print(f.read())

27367283
b'uction_quarter":4,"local_utm_crs":"EPSG:32619"}}]}'


### Load wind dataset

In [6]:
wind_geojson = load_dataset(WIND_GEOJSON_PATH)

In [7]:
wind_h3_cells = turbine_points_to_h3(wind_geojson, H3_RESOLUTION)
wind_df = build_grid_dataframe(generate_conus_h3_cells(H3_RESOLUTION), wind_h3_cells)


In [8]:
wind_df['has_turbine'].value_counts()

,count
has_turbine,
0,19314241
1,41871


### Load solar dataset

In [9]:
def solar_polygons_to_h3(gdf: gpd.GeoDataFrame, resolution: int) -> set[str]:
    """Find all H3 cells that overlap any solar farm polygon."""
    h3_indices = set()
    for _, row in gdf.iterrows():
        geom = row.geometry

        # h3.polygon_to_cells() fills a polygon with H3 cells
        # Need to convert shapely coords to h3's (lat, lng) format
        exterior = [(y, x) for x, y in geom.exterior.coords]
        holes = [
            [(y, x) for x, y in hole.coords]
            for hole in geom.interiors
        ]

        poly = h3.LatLngPoly(exterior, *holes)
        cells = h3.polygon_to_cells(poly, res=resolution)
        h3_indices.update(cells)

    return h3_indices

In [10]:
def build_grid_dataframe_solar(all_cells: set[str], solar_cells: set[str]) -> pd.DataFrame:
    cells = list(all_cells)
    centers = [h3.cell_to_latlng(c) for c in cells]
    return pd.DataFrame({
        "h3_index": cells,
        "lat": [c[0] for c in centers],
        "lng": [c[1] for c in centers],
        "has_solar": pd.array([c in solar_cells for c in cells], dtype="int8"),
    })

In [11]:
# Raise H3 resolution to 8 because solar farms are small
H3_RESOLUTION = 8

solar_geojson = load_dataset(SOLAR_GEOJSON_PATH)
solar_h3_cells = solar_polygons_to_h3(solar_geojson, H3_RESOLUTION)
solar_df = build_grid_dataframe_solar(generate_conus_h3_cells(H3_RESOLUTION), solar_h3_cells)
solar_df.head()

solar_df = solar_df.rename(columns={"has_turbine": "has_solar"})


In [12]:
solar_df['has_solar'].value_counts()

,count
has_solar,
0,19353142
1,2970


### Add exog variables

In [13]:
import polars as pl

wind_df = pl.from_pandas(wind_df)
solar_df = pl.from_pandas(solar_df)

### Combine dfs and filter for only has_turbine and has_solar

In [14]:
combined_df = wind_df.join(solar_df, on="h3_index", how="full")


In [15]:
combined_df = combined_df.with_columns(pl.col('has_turbine').cast(pl.Boolean()))
combined_df = combined_df.with_columns(pl.col('has_solar').cast(pl.Boolean()))

In [16]:
filtered_df = combined_df.filter((pl.col('has_turbine') == True) | (pl.col('has_solar') == True))
filtered_df.shape

(44830, 8)

### Sample negatives

In [17]:
SAMPLE_SIZE = filtered_df.shape[0] * 3
SAMPLE_SIZE

134490

In [18]:
sampled_df = combined_df.filter(pl.col('has_turbine') == False | pl.col('has_solar')).sample(SAMPLE_SIZE)


### Combine dataframes

In [19]:
final_df = pl.concat([filtered_df, sampled_df])
final_df.head()

h3_index,lat,lng,has_turbine,h3_index_right,lat_right,lng_right,has_solar
str,f64,f64,bool,str,f64,f64,bool
"""882b8d2187fffff""",43.197447,-73.054847,false,"""882b8d2187fffff""",43.197447,-73.054847,true
"""88489b8317fffff""",31.411722,-98.399186,true,"""88489b8317fffff""",31.411722,-98.399186,false
"""8826dbd749fffff""",32.615384,-100.670134,true,"""8826dbd749fffff""",32.615384,-100.670134,false
"""8826c5ce61fffff""",36.582413,-97.768362,true,"""8826c5ce61fffff""",36.582413,-97.768362,false
"""88260cd2e5fffff""",42.529763,-95.324487,true,"""88260cd2e5fffff""",42.529763,-95.324487,false


In [27]:
from tqdm.auto import tqdm

In [32]:
DATA_DIR = os.path.abspath(os.path.join(os.path.dirname(""), "..", "data"))
os.makedirs(DATA_DIR, exist_ok=True)


def centroids_to_ee_fc(df: pl.DataFrame) -> ee.FeatureCollection:
    """Convert H3 centroid lat/lng to an ee.FeatureCollection of points."""
    features = []
    for row in df.select("h3_index", "lat", "lng").iter_rows(named=True):
        feat = ee.Feature(
            ee.Geometry.Point([row["lng"], row["lat"]]),
            {"h3_index": row["h3_index"]},
        )
        features.append(feat)
    return ee.FeatureCollection(features)


def build_gee_feature_image() -> ee.Image:
    """Build a single image containing all GEE-derived bands."""
    dem = ee.Image("USGS/SRTMGL1_003")
    terrain = ee.Terrain.products(dem)

    roughness = dem.reduceNeighborhood(
        reducer=ee.Reducer.stdDev(),
        kernel=ee.Kernel.square(3, "pixels"),
    ).rename("roughness")

    era5 = (
        ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR")
        .select(["u_component_of_wind_10m", "v_component_of_wind_10m"])
    )

    def wind_speed(img):
        u = img.select("u_component_of_wind_10m")
        v = img.select("v_component_of_wind_10m")
        return u.pow(2).add(v.pow(2)).sqrt().rename("wind_speed_10m")

    speeds = era5.map(wind_speed)
    mean_speed = speeds.mean().rename("wind_10m_avg")
    std_speed = speeds.reduce(ee.Reducer.stdDev()).rename("wind_10m_std")

    nlcd = ee.Image("USGS/NLCD_RELEASES/2021_REL/NLCD/2021").select("landcover")
    pop = (
        ee.ImageCollection("CIESIN/GPWv411/GPW_Population_Density")
        .sort("system:time_start", False)
        .first()
        .select("population_density")
    )

    return (
        terrain
        .addBands(roughness)
        .addBands(mean_speed)
        .addBands(std_speed)
        .addBands(nlcd)
        .addBands(pop)
    )


def sample_gee_features(fc: ee.FeatureCollection, feature_image: ee.Image) -> dict:
    """Sample all GEE features for one batch with retry/backoff."""
    sampled = feature_image.sampleRegions(collection=fc, scale=30, geometries=False)

    last_error = None
    for attempt in range(4):
        try:
            results = sampled.getInfo()
            break
        except Exception as exc:
            last_error = exc
            if attempt == 3:
                raise
            sleep_s = 2 ** attempt
            print(f"GEE request failed on attempt {attempt + 1}/4; retrying in {sleep_s}s")
            time.sleep(sleep_s)
    else:
        raise last_error

    forest_codes = {41, 42, 43}
    crop_codes = {81, 82}
    urban_codes = {21, 22, 23, 24}
    grass_codes = {71}
    water_codes = {11}

    out = {}
    for feat in results["features"]:
        p = feat["properties"]
        lc = p.get("landcover", 0)
        out[p["h3_index"]] = {
            "h3_elev_mean": p.get("elevation"),
            "h3_slope_mean_deg": p.get("slope"),
            "h3_aspect_mean_deg": p.get("aspect"),
            "h3_roughness": p.get("roughness"),
            "h3_wind_10m_avg": p.get("wind_10m_avg"),
            "h3_wind_10m_std": p.get("wind_10m_std"),
            "h3_land_cover_code": lc,
            "h3_cover_is_forest": int(lc in forest_codes),
            "h3_cover_is_crop": int(lc in crop_codes),
            "h3_cover_is_urban": int(lc in urban_codes),
            "h3_cover_is_grass": int(lc in grass_codes),
            "h3_cover_is_water": int(lc in water_codes),
            "h3_pop_density_km2": p.get("population_density"),
        }
    return out


def fetch_transmission_distances(df: pl.DataFrame) -> dict:
    hifld_path = os.path.join(DATA_DIR, "transmission_lines.geojson")

    if not os.path.exists(hifld_path):
        print("Downloading HIFLD transmission lines (this may take a minute)...")
        url = (
            "https://services1.arcgis.com/Hp6G80Pky0om6HgA/arcgis/rest/services/"
            "Transmission_Lines/FeatureServer/0/query"
            "?where=1%3D1&outFields=*&f=geojson&resultRecordCount=50000"
        )
        all_features = []
        offset = 0
        while True:
            resp = requests.get(url + f"&resultOffset={offset}", timeout=120)
            data = resp.json()
            feats = data.get("features", [])
            if not feats:
                break
            all_features.extend(feats)
            print(f"  Downloaded {len(all_features)} features...")
            offset += len(feats)
            if len(feats) < 50000:
                break

        geojson = {"type": "FeatureCollection", "features": all_features}
        with open(hifld_path, "w") as f:
            json.dump(geojson, f)
        print(f"Saved {len(all_features)} transmission line features.")

    print("Loading transmission lines...")
    gdf = gpd.read_file(hifld_path)

    coords = []
    for geom in gdf.geometry:
        if geom is None:
            continue
        if geom.geom_type == "MultiLineString":
            for line in geom.geoms:
                coords.extend(line.coords)
        elif geom.geom_type == "LineString":
            coords.extend(geom.coords)
    coords = np.array(coords)
    print(f"  {len(coords)} transmission line vertices")

    coords_rad = np.radians(coords[:, ::-1])
    tree = cKDTree(coords_rad)

    lats = df["lat"].to_numpy()
    lngs = df["lng"].to_numpy()
    query_rad = np.radians(np.column_stack([lats, lngs]))
    dists, _ = tree.query(query_rad)
    dists_km = dists * 6371.0

    h3_indices = df["h3_index"].to_list()
    return {
        h3_indices[i]: {"h3_dist_to_transmission_km": float(dists_km[i])}
        for i in range(len(h3_indices))
    }


def fetch_road_distances(df: pl.DataFrame) -> dict:
    roads_dir = os.path.join(DATA_DIR, "tiger_roads")
    roads_shp = os.path.join(roads_dir, "tl_2023_us_primaryroads.shp")

    if not os.path.exists(roads_shp):
        print("Downloading TIGER primary roads shapefile...")
        os.makedirs(roads_dir, exist_ok=True)
        url = "https://www2.census.gov/geo/tiger/TIGER2023/PRIMARYROADS/tl_2023_us_primaryroads.zip"
        resp = requests.get(url, timeout=120)
        zip_path = os.path.join(roads_dir, "roads.zip")
        with open(zip_path, "wb") as f:
            f.write(resp.content)
        import zipfile
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(roads_dir)
        print("Extracted roads shapefile.")

    print("Loading roads...")
    gdf = gpd.read_file(roads_shp)

    coords = []
    for geom in gdf.geometry:
        if geom is None:
            continue
        if geom.geom_type == "MultiLineString":
            for line in geom.geoms:
                coords.extend(line.coords)
        elif geom.geom_type == "LineString":
            coords.extend(geom.coords)
    coords = np.array(coords)
    print(f"  {len(coords)} road vertices")

    coords_rad = np.radians(coords[:, ::-1])
    tree = cKDTree(coords_rad)

    lats = df["lat"].to_numpy()
    lngs = df["lng"].to_numpy()
    query_rad = np.radians(np.column_stack([lats, lngs]))
    dists, _ = tree.query(query_rad)
    dists_km = dists * 6371.0

    h3_indices = df["h3_index"].to_list()
    return {
        h3_indices[i]: {"h3_dist_to_major_road_km": float(dists_km[i])}
        for i in range(len(h3_indices))
    }


GEE_BATCH_SIZE = int(os.getenv("GEE_BATCH_SIZE", "1000"))


def enrich_dataframe(df: pl.DataFrame) -> pl.DataFrame:
    n = len(df)
    n_batches = (n + GEE_BATCH_SIZE - 1) // GEE_BATCH_SIZE

    elev_data = {}
    wind_data = {}
    lc_data = {}
    pop_data = {}
    feature_image = build_gee_feature_image()

    print(f"=== GEE enrichment: {n} rows in {n_batches} batches of {GEE_BATCH_SIZE} ===")
    for i in tqdm(range(0, n, GEE_BATCH_SIZE), desc="GEE batches", total=n_batches):
        batch_start = time.time()
        batch_df = df.slice(i, GEE_BATCH_SIZE)
        fc = centroids_to_ee_fc(batch_df)
        batch_data = sample_gee_features(fc, feature_image)

        for h3_index, values in batch_data.items():
            elev_data[h3_index] = {
                k: v for k, v in values.items()
                if k in {"h3_elev_mean", "h3_slope_mean_deg", "h3_aspect_mean_deg", "h3_roughness"}
            }
            wind_data[h3_index] = {
                k: v for k, v in values.items()
                if k in {"h3_wind_10m_avg", "h3_wind_10m_std"}
            }
            lc_data[h3_index] = {
                k: v for k, v in values.items()
                if k.startswith("h3_land_cover") or k.startswith("h3_cover_")
            }
            pop_data[h3_index] = {
                "h3_pop_density_km2": values.get("h3_pop_density_km2")
            }

        print(
            f"  Batch {i // GEE_BATCH_SIZE + 1}/{n_batches}: "
            f"{len(batch_df)} rows in {time.time() - batch_start:.1f}s"
        )

    print("\n=== 5/6 Distance to Transmission Lines (HIFLD) ===")
    trans_data = fetch_transmission_distances(df)

    print("\n=== 6/6 Distance to Major Roads (TIGER) ===")
    road_data = fetch_road_distances(df)

    all_indices = df["h3_index"].to_list()
    records = []
    for idx in tqdm(all_indices, desc="Merging features"):
        row = {"h3_index": idx}
        for d in [elev_data, wind_data, lc_data, pop_data, trans_data, road_data]:
            row.update(d.get(idx, {}))
        records.append(row)

    exog_df = pl.DataFrame(records)
    enriched = df.join(exog_df, on="h3_index", how="left")

    print(f"\nDone! Shape: {enriched.shape}")
    print(f"Columns: {enriched.columns}")
    return enriched

In [33]:
ee.Authenticate()
ee.Initialize(project='renewably')

In [30]:
final_df.head()

h3_index,lat,lng,has_turbine,h3_index_right,lat_right,lng_right,has_solar
str,f64,f64,bool,str,f64,f64,bool
"""882b8d2187fffff""",43.197447,-73.054847,false,"""882b8d2187fffff""",43.197447,-73.054847,true
"""88489b8317fffff""",31.411722,-98.399186,true,"""88489b8317fffff""",31.411722,-98.399186,false
"""8826dbd749fffff""",32.615384,-100.670134,true,"""8826dbd749fffff""",32.615384,-100.670134,false
"""8826c5ce61fffff""",36.582413,-97.768362,true,"""8826c5ce61fffff""",36.582413,-97.768362,false
"""88260cd2e5fffff""",42.529763,-95.324487,true,"""88260cd2e5fffff""",42.529763,-95.324487,false


In [39]:
df_with_exog = enrich_dataframe(final_df)

=== GEE enrichment: 179320 rows in 90 batches of 2000 ===


GEE batches:   0%|          | 0/90 [00:00<?, ?it/s]

GEE request failed on attempt 1/4; retrying in 1s


KeyboardInterrupt: 

In [ ]:
df_with_exog.write_csv('/content/df_with_exog.csv')

In [ ]:
ee.Authenticate()
ee.Initialize(project='renewably')

In [3]:
tasks = ee.batch.Task.list()
for task in tasks[:20]:
    status = task.status()
    print(
        status.get("description"),
        status.get("state"),
        status.get("creation_timestamp_ms"),
        status.get("update_timestamp_ms"),
    )

EEException: Request had insufficient authentication scopes.